In [ ]:
import polars as pl
import seaborn as sns
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from modules.design import wrap_and_center

mpl.rcParams['font.family'] = 'Liberation Serif'


# Import csv

In [ ]:
instruction_profile_file = "../data/benchmark_profiles.csv"
instruction_profile_df = pl.read_csv(instruction_profile_file, separator=";", decimal_comma=True, quote_char="\"")
instruction_profile_df = instruction_profile_df.with_columns(name=pl.col("Embench IOT Name").str.replace("nettle-sha256", "sha256"))

instruction_profile_df = instruction_profile_df.with_columns(
    Branching=pl.col("Branching") * 100.0,
    Memory=pl.col("Memory") * 100.0,
    Compute=pl.col("Compute") * 100.0,
)


# Process Raw Data

In [ ]:
metrics = ["Branching", "Memory", "Compute"]

EMBENCH_TRANSLATION_DICT = {
    "aha-mont64": "mont64",
    "crc32": "crc_32",
    "md5sum": "md5",
}

def translate(s):
    return EMBENCH_TRANSLATION_DICT.get(s, s)

tud_benchmarks = set(["mont64", "crc_32", "huffbench", "matmult-int", "sha256", "nsichneu", "slre", "statemate", "ud", "md5", "tarfind", "xgboost"])
tum_benchmarks = set(["mont64", "crc_32", "depthconv", "edn", "huffbench", "matmult-int", "md5", "sha256", "qrduino", "statemate", "tarfind", "ud"])
all_benchmarks = set(instruction_profile_df["name"].to_list())

tum_order = ["mont64", "sha256", "md5", "crc_32", "ud", "qrduino", "tarfind", "huffbench", "statemate", "matmult-int", "edn", "depthconv"]

p_df = instruction_profile_df.with_columns(name=pl.col("name").map_elements(translate)).select(["name", "Branching", "Memory", "Compute"])
p_df =  p_df.with_columns([
    (pl.col(c).rank("min") / pl.col(c).len() * 100).alias(f"{c}_pct")
    for c in metrics
])

# Original table
This is a recreation of the original table from the paper.
It shows the benchmarks and their profile.

In [ ]:
# Extract names
names = p_df["name"].to_list()
columns = ['Branching', 'Memory', 'Compute']
colors = {'low': '#6bcb77', 'mid': '#ffd93d', 'high': '#ff6b6b'}

value_matrix = np.column_stack([p_df[col].to_numpy() for col in columns])
n = value_matrix.shape[0]

percentiles = {
    j: (np.percentile(value_matrix[:, j], 25), np.percentile(value_matrix[:, j], 75))
    for j in range(len(columns))
}

color_matrix = np.empty(value_matrix.shape, dtype=object)
for i in range(n):
    for j in range(len(columns)):
        val = value_matrix[i, j]
        p25, p75 = percentiles[j]
        if val <= p25:
            color_matrix[i, j] = colors['low']
        elif val >= p75:
            color_matrix[i, j] = colors['high']
        else:
            color_matrix[i, j] = colors['mid']

fig, ax = plt.subplots(figsize=(7, 10))
for i in range(value_matrix.shape[0]):
    for j in range(value_matrix.shape[1]):
        rect = plt.Rectangle([j, i], 1, 1, facecolor=color_matrix[i, j], edgecolor='black')
        ax.add_patch(rect)
        ax.text(j + 0.5, i + 0.5, f"{value_matrix[i, j]:.2f}", va='center', ha='center', fontsize=12, color='black')

ax.set_xlim(0, len(columns))
ax.set_ylim(0, n)
ax.set_xticks(np.arange(len(columns)) + 0.5)
ax.tick_params(axis='x', top=True, labeltop=True, bottom=False, labelbottom=False)
ax.set_xticklabels(columns, fontsize=13)
ax.set_yticks(np.arange(n) + 0.5)
ax.set_yticklabels(names, fontsize=12)
ax.invert_yaxis()
ax.set_xlabel("", fontsize=14)
ax.set_ylabel("", fontsize=14)
ax.set_title("Percentile-Based Heatmap of DataFrame Columns", fontsize=15, pad=15)


legend_elements = [
    Patch(facecolor=colors['low'], edgecolor='black', label='≤ 25th percentile'),
    Patch(facecolor=colors['mid'], edgecolor='black', label='25th–75th percentile'),
    Patch(facecolor=colors['high'], edgecolor='black', label='≥ 75th percentile'),
]
ax.legend(handles=legend_elements, loc='center', bbox_to_anchor=(0.5, -0.03), title="", ncols=3)

plt.tight_layout()
plt.show()

# Heatmap Benchmark Grouping
This

In [ ]:
sns.set_theme(style="whitegrid", font="Liberation Serif", font_scale=1.3)
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4.5))

# A custom order of both the tum and tud benchmarks
custom_order=["mont64", "sha256", "md5", "crc_32", "depthconv", "qrduino", "ud", "xgboost", "tarfind", "huffbench", "statemate", "matmult-int", "edn", "nsichneu", "slre"]
assert set(custom_order) == tum_benchmarks | tud_benchmarks
ordered_df = p_df.filter(pl.col("name").is_in(tud_benchmarks | tum_benchmarks)).sort(pl.col("name").cast(pl.Enum(custom_order)))

percentile_matrix = ordered_df.select([f"{c}_pct" for c in metrics]).to_numpy()
annot_matrix = ordered_df.select(metrics).to_numpy()
xticklabels = ordered_df["name"].to_list()

sns.heatmap(
    np.transpose(percentile_matrix),
    ax=ax,
    annot=np.transpose(annot_matrix),
    annot_kws={"weight": "heavy"},
    fmt=".0f",
    square=True,
    cmap="RdYlGn_r",
    vmin=0, vmax=100,
    cbar_kws={"label": "Percentile within Operation Type", "location": "top", "pad": 0.14, "aspect": 30},
    xticklabels=xticklabels,
    yticklabels=metrics,
    cbar=True,
)

### X AXIS ###
annotated_xlabels = []
for label in ax.get_xticklabels():
    t = label.get_text()
    if t in tud_benchmarks.difference(tum_benchmarks):
        annotated_xlabels.append(f"{t}")
        label.set_fontweight("bold")
    elif t in tum_benchmarks.difference(tud_benchmarks):
        annotated_xlabels.append(f"{t}")
        label.set_color("#808080")
    else:
        annotated_xlabels.append(t)

ax.set_xticklabels(annotated_xlabels)

# rotation
ax.tick_params(axis="both", rotation=45)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_horizontalalignment("right")
    label.set_verticalalignment("top")


### SECONDARY AXIS ###
# The divider vlines and the ticks between two lines
cat_divider = [2, 5, 8, 10, 13]
cat_tick_locations = [1, 3.5, 6.5, 9, 11.5, 14]
cat_tick_labels = ["Pure Compute", "Compute", "Balanced", "Branching", "Memory", "Memory + Branching"]

# Adding the divider lines
for i in cat_divider:
    ax.vlines(i, *ax.get_ylim(), color="black", linewidth=1.65)

# Secondary Axis on top to show the benchmark categories
sec = ax.secondary_xaxis(location="top")
sec.set_xticks(cat_tick_locations)
sec.set_xticklabels(wrap_and_center(cat_tick_labels, 10))
sec.tick_params(axis="x", pad=14)
for label in sec.get_xticklabels():
    label.set_verticalalignment("center")

for tick in ax.yaxis.get_major_ticks():
    tick.tick1line.set_visible(True)

for tick in ax.xaxis.get_major_ticks():
    tick.tick1line.set_visible(True)

plt.xlabel("Benchmarks")
plt.ylabel("Ration of Operation Types [%]")

cb = ax.collections[0].colorbar
cb.outline.set_edgecolor("black")
cb.outline.set_linewidth(1.3)

plt.tight_layout(h_pad=0.2)
plt.savefig("figures/one_column_profile_cbar_top.pdf", bbox_inches="tight", format="pdf")
